## Polypharmacy classification
- Polypharmacy: ≥5 drugs
- Hyperpolypharmacy: ≥10 drugs

In [2]:
import pandas as pd

# Load both files
df_10 = pd.read_csv(r"C:\Users\laiar\Desktop\DATOS_PADRIS_nets\10_num_visites_i_far_Final.csv")
schiz_patients = pd.read_csv(r'C:\Users\laiar\Desktop\Classe IDIBAPS\data_sintetica\Data-sintetica\Data-sintetica\src\csv merged\cohort_schizophrenia_sociodemografica.csv')

# Filter visits to only schizophrenia patients
df_10 = df_10[df_10['idCas'].isin(schiz_patients['idCas'])]

print(f"Total patients: {df_10['idCas'].nunique()}")

Total patients: 9969


In [4]:
df_10["Origen"].unique()

array([2019, 2018])

In [3]:
def classify_polypharmacy(n):
    if n < 5:
        return 'No polypharmacy'
    elif n < 10:
        return 'Polypharmacy'
    else:
        return 'Hyperpolypharmacy'

df_10['polypharmacy_class'] = df_10['F10_Farmacs'].apply(classify_polypharmacy)
print(df_10['polypharmacy_class'].value_counts())

polypharmacy_class
No polypharmacy      8679
Polypharmacy         6948
Hyperpolypharmacy    4297
Name: count, dtype: int64


In [5]:
df_10['polypharmacy_class'] = df_10['F10_Farmacs'].apply(classify_polypharmacy)
print(df_10.groupby('Origen')['polypharmacy_class'].value_counts())

Origen  polypharmacy_class
2018    No polypharmacy       4338
        Polypharmacy          3482
        Hyperpolypharmacy     2135
2019    No polypharmacy       4341
        Polypharmacy          3466
        Hyperpolypharmacy     2162
Name: count, dtype: int64


In [8]:
# Pivot so each patient has one row with 2018 and 2019 classification
pivot = df_10.groupby(['idCas', 'Origen'])['polypharmacy_class'].first().unstack('Origen')
pivot.columns = ['class_2018', 'class_2019']

# Only patients present in both years
both_years = pivot.dropna()

# Flag changes
both_years['changed'] = both_years['class_2018'] != both_years['class_2019']

print(f"Patients in both years: {len(both_years)}")
print(f"Changed category: {both_years['changed'].sum()}")
print()
print(both_years.groupby(['class_2018', 'class_2019']).size())

Patients in both years: 9955
Changed category: 3078

class_2018         class_2019       
Hyperpolypharmacy  Hyperpolypharmacy    1474
                   No polypharmacy       127
                   Polypharmacy          534
No polypharmacy    Hyperpolypharmacy     100
                   No polypharmacy      3356
                   Polypharmacy          882
Polypharmacy       Hyperpolypharmacy     588
                   No polypharmacy       847
                   Polypharmacy         2047
dtype: int64


C:\Users\laiar\AppData\Local\Temp\ipykernel_36288\2032536195.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  both_years['changed'] = both_years['class_2018'] != both_years['class_2019']


In [9]:
df_10 = df_10.merge(both_years[['class_2018', 'class_2019', 'changed']], on='idCas', how='right')

In [11]:
df_10[['idCas', 'class_2018', 'class_2019', 'changed']].to_csv(r"C:\Users\laiar\Desktop\Classe IDIBAPS\data_sintetica\Data-sintetica\Data-sintetica\src\csv merged\10_polypharmacy_changes.csv", index=False)